# 00 · Load the match data

Reads every provided asset into numpy/pandas, aligned so that **row `i` = video frame `i` = row `i` of a submission**.

## Setup — run once

From the repository root, install the dependencies for the main notebooks and pipeline:

```bash
python -m pip install -r requirements.txt
```

Optional: install the heavier dependencies only if you use the raw-video/George ball-observation appendix:

```bash
python -m pip install -r requirements.txt -r ballfill/requirements.txt
```

Restart the notebook kernel after installation, then run this notebook first.

## What this notebook loads

The loading code lives in `possession_lab/baseline_pipeline.py`. It creates the shared, frame-aligned table used by the later notebooks.

### Data notes

- Coordinates are in **metres**, origin at a pitch corner: x along the length (0–105.3), y across (0–68).
- `NaN` = not tracked (ball or player).
- First run parses the tracking (~3 s) and caches it in `cache/` (gitignored, never commit it).

The optional George artifact `data/ball/BAR-ATH_ball.csv` is a separate, frame-aligned observation table with source and uncertainty metadata. It is validated by the reusable adapter before feature construction; it is not generated here or treated as ground truth.

In [1]:
import os, sys
from pathlib import Path

ROOT = Path.cwd().resolve()
while not (ROOT / 'possession_lab').exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
DATA_CANDIDATES = []
if os.environ.get('MTAH_DATA_DIR'):
    DATA_CANDIDATES.append(Path(os.environ['MTAH_DATA_DIR']))
DATA_CANDIDATES.extend([ROOT / 'data', ROOT.parent / 'data'])
DATA = next((candidate.resolve() for candidate in DATA_CANDIDATES if (candidate / 'tactical').exists()), None)
if DATA is None:
    raise FileNotFoundError('Set MTAH_DATA_DIR to the folder containing tactical/')
os.environ['MTAH_DATA_DIR'] = str(DATA)
sys.path.insert(0, str(ROOT))

import numpy as np
import pandas as pd
from possession_lab.baseline_pipeline import add_match_time_columns, load_supplied_frame_table

%load_ext autoreload
%autoreload 2
print('project root:', ROOT)
print('data dir:', DATA)

project root: /Users/luke/Desktop/barca-hack-repo-staging
data dir: /Users/luke/Desktop/barca-hack-repo-staging/data


## Load

In [2]:
m, load_report = load_supplied_frame_table(data_dir=DATA)
m = add_match_time_columns(m)
print('loaded frame table:', m.shape)
print(load_report)

loaded frame table: (199319, 150)
{'raw_shape_after_empty_drop': (199319, 145), 'dropped_all_missing_atd_columns': ['44_x', '44_y', '44_d'], 'dropped_all_missing_on_graded_frames': ['22_x', '22_y', '22_d', '23_x', '23_y', '23_d'], 'n_frames': 199319, 'frame_range': (0, 199318)}


## What's in the frame table `m`

| attribute | shape | meaning |
|---|---|---|
| `ball_x`, `ball_y`, `ball_z` | N columns | supplied ball coordinates in metres; NaN means not observed |
| `<slot>_x`, `<slot>_y` | 46 track slots | player coordinates; a slot is a tracklet, **not a person** |
| `<slot>_s`, `<slot>_d` | optional | supplied player speed/direction when available |
| `frame`, `frame_id`, `match_half` | N rows | zero-based submission frame plus source/time alignment |
| `video_time_s`, `match_clock_s` | N rows | derived time indices used for forward/backward audits |
| public GT | separate CSV | H/A/D on the five-minute window; **checking only, never tuning** |

In [3]:
key_columns = [
    column for column in [
        'frame', 'frame_id', 'period_id', 'ball_x', 'ball_y', 'ball_z',
        'video_time_s', 'match_half', 'match_clock_status'
    ] if column in m.columns
]
print('key columns:', key_columns)
print('shape:', m.shape)
m[key_columns].head(3)

key columns: ['frame', 'frame_id', 'ball_x', 'ball_y', 'ball_z', 'video_time_s', 'match_half', 'match_clock_status']
shape: (199319, 150)


,frame,frame_id,ball_x,ball_y,ball_z,video_time_s,match_half,match_clock_status
0,0,1,NaN,NaN,NaN,0.00,1H,in-play timeline
1,1,2,NaN,NaN,NaN,0.04,1H,in-play timeline
2,2,3,NaN,NaN,NaN,0.08,1H,in-play timeline


## A per-frame table (handy for quick looks)

## Frame-to-match-time utility

Frame time is part of the shared data interface, not a possession heuristic. The helper adds:

- video_time_s: exact tactical video time at the supplied frame rate;
- half_elapsed_s: signed time from the verified half kickoff;
- match_clock_s: kickoff-anchored analytical clock, with 0:00 at first-half kickoff and 45:00 at second-half kickoff;
- match_clock_status: makes the graded second-half pre-kickoff interval explicit.

The analytical clock is for reproducible lookup and plot annotation. It is not a claim about the referee’s displayed stoppage-time clock.

In [4]:
from possession_lab.baseline_pipeline import add_match_time_columns

time_index = add_match_time_columns(m[['frame']].copy())
display(time_index.loc[time_index.match_half.ne(time_index.match_half.shift()), ['frame', 'match_half', 'video_time_s', 'half_elapsed_s', 'match_clock_s', 'match_clock_status']])

,frame,match_half,video_time_s,half_elapsed_s,match_clock_s,match_clock_status
0,0,1H,0.00,-447.40,-447.40,in-play timeline
110808,110808,2H,4432.32,-117.36,2582.64,2H pre-kickoff


In [5]:
ball_seen = m[['ball_x', 'ball_y']].notna().all(axis=1)
player_columns = [column for column in m.columns if column.endswith('_x') and column.split('_', 1)[0].isdigit()]
frames = pd.DataFrame({
    'frame': m['frame'],
    'match_half': m['match_half'],
    'ball_x': m['ball_x'], 'ball_y': m['ball_y'],
    'ball_seen': ball_seen,
    'n_player_tracks': m[player_columns].notna().sum(axis=1),
})
public_gt_path = ROOT / 'evaluation' / 'gt' / 'BAR-ATH_possession_gt_public.csv'
if public_gt_path.exists():
    public_gt = pd.read_csv(public_gt_path).set_index('frame')['label']
    frames['gt_public'] = public_gt.reindex(frames['frame']).to_numpy()
display(frames.loc[11185:11195])

,frame,match_half,ball_x,ball_y,ball_seen,n_player_tracks,gt_public
11185,11185,1H,NaN,NaN,False,20,A
11186,11186,1H,NaN,NaN,False,20,A
11187,11187,1H,0.50431,0.48090,True,20,A
11188,11188,1H,0.51057,0.48349,True,20,A
11189,11189,1H,0.51691,0.48613,True,20,A
11190,11190,1H,0.52331,0.48880,True,20,A
11191,11191,1H,0.52974,0.49149,True,20,A
11192,11192,1H,0.53619,0.49419,True,20,A
11193,11193,1H,0.54263,0.49689,True,21,A
11194,11194,1H,0.54904,0.49959,True,21,A


## Downstream audit views

This notebook intentionally stops after loading the shared match object, its frame table, and the frame-to-time utility.
Pitch views, missing-ball analysis, public-window checks, and Smart Tagging conclusions live in **01 — Problem, data, and findings**.
To reuse the submission, update only `MTAH_DATA_DIR` (or the fallback data path) here; the later notebooks consume the same interface.


## Organiser handoff

**Problem → idea → construction → evaluation.** Possession truth is not directly observed, so every later stage must consume the same frame-aligned tracking/event interface. This notebook constructs that reproducible input boundary; notebooks 01–06 derive observations, scenarios, anchors, and transition evidence, while 08 runs the final full-match pipeline and 09 compares alternatives. Ground truth is read only by the evaluation notebooks after prediction.